# NB24 — Phase A : calibration des taux de but `alpha` (moteur Poisson buteur)

À relancer chaque fois que les cotes buteur changent (pré-tournoi OU en cours de
tournoi). Calibre :
- `alpha` (buts/match intrinsèque GO-FORWARD) par buteur listé,
- `alpha_field` (taux des buteurs de FOND génériques, un par équipe),

pour que la fréquence simulée de *meilleur buteur* (convention dead-heat) colle aux
cotes buteur dé-viggées (Shin). Écrit le résultat dans la colonne `alpha` de
`data/CDM_2026_goal_scorer_and_favorite.csv` (buteurs + ligne `field`).

**Calibration IN-PLAY consistante** : on utilise les buts DÉJÀ marqués (`current_goals`)
et les matchs RESTANTS (poule restante via `CDM_2026.csv` + knockout simulé), de sorte
que `current_goals + Poisson(alpha × M_restants)` reproduise la cote courante — **pas de
double comptage** (un joueur en tête a une cote courte à cause de ses buts déjà marqués,
pas seulement de son taux). L'échelle est ancrée par `G_FAV` = buts TOTAUX attendus du
favori (réalisme), ce qui préserve la dynamique de *catch-up* (équipe éliminée →
`M_restants=0` → total gelé).

⚠️ Hypothèse : le tableau knockout est encore OUVERT (poules / début knockout sans
éliminations connues). En knockout avancé, la composante knockout simulée des équipes
déjà éliminées serait surestimée (à raffiner si besoin).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy')
    !pip install shin numba
else:
    PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / 'data'

from mpp_project.core import calculate_true_outcome_probas_from_odds
from mpp_project.scorer_model import (
    simulate_team_matches_played, calibrate_scorer_alphas, load_scorer_players,
    compute_group_matches_remaining,
)

MARKET = DATA_DIR / 'CDM_2026_goal_scorer_and_favorite.csv'
ODDS = DATA_DIR / 'CDM_2026_group_stage_odds.csv'
TOURNOI = DATA_DIR / 'CDM_2026.csv'   # pour les matchs de poule RESTANTS (in-play)

# Paramètres de calibration
# G_FAV = buts TOTAUX (fin de tournoi) attendus du favori du marché. La part FUTURE
# calibrée = G_FAV - ses buts déjà marqués -> alpha = taux GO-FORWARD, sans double
# comptage. Le moteur terminal fait current_goals + Poisson(alpha × M_restants).
# Soulier d'or ~ 6-8 buts ; ajuster si les E[buts totaux] (cellule 4) semblent hauts.
G_FAV = 7.0
N_RUNS = 20000       # tournois simulés pour la distribution des matchs joués
BETA = 0.89          # amortissement force conditionnelle (cf. NB23, valeur à jour)
N_POISSON = 5        # tirages Poisson par tournoi

In [2]:
# 1. Cible = P(meilleur buteur) dé-viggée (Shin) sur les cotes buteur COURANTES
df_market = pd.read_csv(MARKET)
df_odds = pd.read_csv(ODDS)
df_tournoi = pd.read_csv(TOURNOI)
players, autre_gain, _ = load_scorer_players(df_market)
target = calculate_true_outcome_probas_from_odds(players['cote'].values.astype(float))
print(f'{len(players)} buteurs ({int(players.is_ghost.sum())} ghosts), cible Shin somme={target.sum():.3f}')

team_to_id = {str(t).strip().lower(): i for i, t in enumerate(df_odds['team'])}
nation_id = np.array([team_to_id[n] for n in players['nation']], dtype=np.int64)
field_nation_id = np.arange(len(df_odds), dtype=np.int64)  # un buteur de fond par équipe

# IN-PLAY : buts DÉJÀ marqués (réels) + matchs de poule RESTANTS (depuis CDM_2026.csv).
current_goals = players['current_goals'].values.astype(float)
group_remaining = compute_group_matches_remaining(df_tournoi, df_odds['team'])
print(f'buts déjà marqués (total listés) : {current_goals.sum():.0f} | '
      f'matchs de poule restants (somme équipes) : {int(group_remaining.sum())}')

44 buteurs (24 ghosts), cible Shin somme=1.000
buts déjà marqués (total listés) : 21 | matchs de poule restants (somme équipes) : 94


In [3]:
# 2. Distribution des matchs FUTURS par équipe (poule restante + knockout)
matches, team_names = simulate_team_matches_played(
    df_odds, n_runs=N_RUNS, beta=BETA, seed=7,
    group_matches_per_team=group_remaining,   # in-play : matchs de poule RESTANTS
)
print('E[matchs restants] top equipes :')
for i in np.argsort(-matches.mean(0))[:5]:
    print(f'  {team_names[i]:>12} {matches[:, i].mean():.2f}')

E[matchs restants] top equipes :
        france 5.34
    angleterre 5.09
       espagne 5.01
     argentine 4.97
      portugal 4.95


In [4]:
# 3. Calibration in-play consistante (current_goals réels + matchs restants)
# alpha tel que current_goals + Poisson(alpha × M_restants) reproduise la cote courante.
alpha, info = calibrate_scorer_alphas(
    target, nation_id, matches.astype(float), field_nation_id,
    current_goals=current_goals, g_fav=G_FAV,
    n_poisson_per_run=N_POISSON, verbose=True, seed=3,
)
print(f"\np={info['p']:.3f}  alpha_field={info['alpha_field']:.3f}  "
      f"MAE={info['mae']:.4f}  rankcorr={info['rankcorr']:.3f}  field_share={info['field_share']:.3f}")

  [calib] p=0.15 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.20 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.25 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.30 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.35 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.40 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.45 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.50 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.55 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.60 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.65 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.70 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)
  [calib] p=0.75 -> meilleur MAE courant 0.0064 (p*=0.15, alpha_field*=0.75)

In [5]:
# 4. Contrôle : cible vs simulé + projection de buts (doit rester réaliste ~6-8 au top)
EM = matches[:, nation_id].mean(0)   # E[matchs RESTANTS] par joueur
ctrl = pd.DataFrame({
    'buteur': players['selection'], 'nation': players['nation'],
    'target': target, 'sim': info['sim_probs'],
    'cg': current_goals, 'alpha': alpha,
    'E[buts futurs]': alpha * EM,
    'E[buts TOTAL]': current_goals + alpha * EM,
}).sort_values('target', ascending=False)
ctrl.head(15).style.format({'target': '{:.3f}', 'sim': '{:.3f}', 'cg': '{:.0f}',
                            'alpha': '{:.3f}', 'E[buts futurs]': '{:.2f}', 'E[buts TOTAL]': '{:.2f}'})

,buteur,nation,target,sim,cg,alpha,E[buts futurs],E[buts TOTAL]
0,kylian_mbappe,france,0.207,0.135,2,0.937,5.00,7.00
1,harry_kane,angleterre,0.194,0.133,2,0.973,4.95,6.95
3,lionel_messi,argentine,0.194,0.216,3,0.997,4.95,7.95
2,erling_haaland,norvege,0.072,0.078,2,0.994,4.26,6.26
37,kai_havertz,allemagne,0.060,0.073,2,0.924,4.15,6.15
5,lamine_yamal,espagne,0.020,0.009,0,0.702,3.52,3.52
16,luis_diaz,colombie,0.020,0.021,1,0.837,3.52,4.52
11,folarin_balogun,etats_unis,0.020,0.035,2,0.809,3.52,5.52
6,vinicius_junior,bresil,0.016,0.017,1,0.729,3.39,4.39
20,mikel_oyarzabal,espagne,0.016,0.008,0,0.678,3.39,3.39


In [6]:
# 5. Écriture dans le CSV (colonne alpha des scorers + ligne field)
sel_to_alpha = {players['selection'].iloc[i]: round(float(alpha[i]), 4) for i in range(len(players))}

def _set_alpha(r):
    cat = str(r['category']).strip().lower()
    sel = str(r['selection']).strip().lower()
    if cat == 'field':
        return round(float(info['alpha_field']), 4)
    if cat == 'scorer' and sel != 'autre':
        return sel_to_alpha.get(sel, '')
    return r['alpha'] if 'alpha' in r and pd.notna(r['alpha']) else ''

df_market['alpha'] = df_market.apply(_set_alpha, axis=1)
df_market.to_csv(MARKET, index=False)
print(f'alpha écrit dans {MARKET.name}. Prêt pour compute_robust_endgame_horizon.')

alpha écrit dans CDM_2026_goal_scorer_and_favorite.csv. Prêt pour compute_robust_endgame_horizon.
